In [17]:
import pandas as pd
import dspy

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from concurrent.futures import ThreadPoolExecutor, as_completed
import tiktoken
import time 
import os 

from tqdm.auto import tqdm
tqdm.pandas()

import os
import sys
sys.path.append("./utils")


from absa_dspy import *
from absa_pipeline import *

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import pprint
from openai import OpenAI

pd.set_option(
    "display.max_colwidth", 12,
    "display.max_rows", 20,
    "display.max_columns", 50,
    "display.precision", 4
)
plt.style.use('./utils/MyStyle.txt')
cm = 1/2.54
palette = ["#3F3517",  '#CE2E31', '#C96F6B', '#CCA464', '#F8D768', '#F0DCD4']

In [ ]:
GPT_MODEL = 'openai/gpt-5-nano'
GPT_API_KEY = ''

In [ ]:
# GROK_MODEL = "xai/grok-4-1-fast-reasoning"
# GROK_API_KEY = ""
# GROK_BASE_URL = "https://api.x.ai/v1"

In [21]:
def make_prompt(review: str) -> str:
    return f"""Extract aspects in the review. For each aspect, assign one category: staff_attitude, service_process, food_drink, pricing_value, location_access, facilities_amenities, cleanliness_maintenance, atmosphere_vibe, natural_environment, activities_events, safety_security, policies_governance, inclusivity_community, overall_experience, other. For each category, classify the sentiment regarding that category as 'positive' | 'negative' | 'neutral'. Avoid “other” when possible. Output unique items only.
 Review: {review}
 Output: Pythonic list of {'category', 'sentiment'} only."""

In [ ]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="",
# api_key = GPT_API_KEY
)

def absa(review_text):
    def make_prompt(review: str) -> str:
        return f"""Extract aspects in the review. For each aspect, assign one category: staff_attitude, service_process, food_drink, pricing_value, location_access, facilities_amenities, cleanliness_maintenance, atmosphere_vibe, natural_environment, activities_events, safety_security, policies_governance, inclusivity_community, overall_experience, other. For each category, classify the sentiment regarding that category as 'positive' | 'negative' | 'neutral'. Avoid “other” when possible. Output unique items only. Review: {review}. Output: Pythonic list of {'category', 'sentiment'} only."""
    
    response = client.chat.completions.create(
        extra_body={},
        # model="microsoft/phi-4-multimodal-instruct",   
        # model="mistralai/mistral-nemo", 
        model='google/gemma-3-12b-it', 
        # model='google/gemma-3-27b-it',
        messages=[
                    {"role": "system", "content": "Aspect-Based Sentiment Analysis (ABSA) involves identifying specific entities (such as a person, product, service, or experience, etc.) mentioned in a text and determining the sentiment expressed toward each entity. You are an expert annotator for Aspect-Based Sentiment Analysis (ABSA) task."},
                    {"role": "user", "content": make_prompt(review_text)}
                    ],
    )

    # response = client.chat.completions.create(
    #     model="gpt-4.1-nano", 
    #     messages=[
    #                 {"role": "system", "content": "Aspect-Based Sentiment Analysis (ABSA) involves identifying specific entities (such as a person, product, service, or experience, etc.) mentioned in a text and determining the sentiment expressed toward each entity. You are an expert annotator for Aspect-Based Sentiment Analysis (ABSA) task."},
    #                 {"role": "user", "content": make_prompt(review_text)}
    #                 ],
    # )



    # usage = response.usage
    # print("Input tokens :", usage.prompt_tokens)
    # print("Output tokens:", usage.completion_tokens)
    # print("Total tokens :", usage.total_tokens)

    output = response.choices[0].message.content
    return output


def run_absa_row(row):
    # if len(row["text"].split(" ")) < 1:
    #     return row["ind"], []
    return row["ind"], absa(row["text"])

In [35]:
data_files = os.listdir('./data/')
result_files = os.listdir('./results/')
jsonl_files = os.listdir('./jsonl_files/')

In [36]:
# for file in tqdm(data_files, total=len(data_files), desc="Processing files"):
#     if file not in result_files:
#         # print(file)
#         df = pd.read_excel('./data/' + file)
#         df = df.loc[:, ['place_id', 'author', 'place_name_y', 'date_parsed', 'text']]
#         df['ind'] = df.index.astype(int)
        
#         results = {}

#         with ThreadPoolExecutor(max_workers=300) as executor:
#             futures = {
#                 executor.submit(run_absa_row, row): idx
#                 for idx, row in df.iterrows()
#             }

#             try:
#                 for future in tqdm(as_completed(futures), total=len(futures), desc=f"Processing {file}"):
#                     try:
#                         ind, res = future.result()
#                         results[ind] = res
#                     except RuntimeError as e:
#                         # Our custom signal: stop everything if daily limit reached
#                         if "Daily limit reached" in str(e):
#                             print("Daily limit hit. Stopping further processing.")
#                             break
#                         else:
#                             # other runtime errors if you want to log them
#                             print("Runtime error:", e)
#             finally:
#                 # Cancel remaining futures (Python 3.9+)
#                 for f in futures:
#                     f.cancel()


#         df["absa_results"] = df["ind"].map(results)
#         df.to_excel('./results/' + file, index=False)
        

In [37]:
for file in tqdm(data_files, total=len(data_files), desc="Processing files"):
    if file not in result_files:
        flag = False
        # print(file)
        df = pd.read_excel('./data/' + file)
        df = df.loc[:, ['place_id', 'author', 'place_name_y', 'date_parsed', 'text']]
        df['ind'] = df.index.astype(int)
        
        results = {}

        with ThreadPoolExecutor(max_workers=500) as executor:
            futures = {
                executor.submit(run_absa_row, row): idx
                for idx, row in df.iterrows()
            }

            try:
                for future in tqdm(as_completed(futures), total=len(futures), desc=f"Processing {file}"):
                    try:
                        ind, res = future.result()
                        results[ind] = res

                    except RuntimeError as e:
                        flag = True
                        # Hard stop if daily limit reached
                        if "Daily limit reached" in str(e):
                            print("Daily limit hit. Stopping further processing.")
                            break

                        # Retry section
                        print(f"Runtime error. Retrying after short pause: {e}")
                        time.sleep(2)

                        # Re-submit the failed job
                        idx = futures[future]
                        row = df.loc[idx]
                        new_future = executor.submit(run_absa_row, row)
                        futures[new_future] = idx

            except Exception as e:
                flag = True
                # Non-runtime errors, retry as well
                print(f"Unexpected error. Retrying after short pause: {e}")
                time.sleep(2)

                idx = futures[future]
                row = df.loc[idx]
                new_future = executor.submit(run_absa_row, row)
                futures[new_future] = idx
            finally:
                # Cancel remaining futures (Python 3.9+)
                for f in futures:
                    f.cancel()

        if flag==False:
            df["absa_results"] = df["ind"].map(results)
            df.to_excel('./results/' + file, index=False)
        

Processing files:   0%|          | 0/5000 [00:00<?, ?it/s]

Processing 962. part_962.xlsx:   0%|          | 0/1215 [00:00<?, ?it/s]

In [ ]:
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="",
)


def absa(review_text):
    def make_prompt(review: str) -> str:
        return f"""Extract aspects in the review. For each aspect, assign one category: staff_attitude, service_process, food_drink, pricing_value, location_access, facilities_amenities, cleanliness_maintenance, atmosphere_vibe, natural_environment, activities_events, safety_security, policies_governance, inclusivity_community, overall_experience, other. For each category, classify the sentiment regarding that category as 'positive' | 'negative' | 'neutral'. Avoid “other” when possible. Output unique items only. Review: {review}. Output: Pythonic list of {'category', 'sentiment'} only."""
    
    response = client.chat.completions.create(
        extra_body={},
        model="google/gemma-3-12b-it",     # or any model you are using
        messages=[
                    {"role": "system", "content": "Aspect-Based Sentiment Analysis (ABSA) involves identifying specific entities (such as a person, product, service, or experience, etc.) mentioned in a text and determining the sentiment expressed toward each entity. You are an expert annotator for Aspect-Based Sentiment Analysis (ABSA) task."},
                    {"role": "user", "content": make_prompt(review_text)}
                    ],
        # max_completion_tokens=128,
        # temperature=0.0,
    )

    # --- Token usage ---
    usage = response.usage
    print("Input tokens :", usage.prompt_tokens)
    print("Output tokens:", usage.completion_tokens)
    print("Total tokens :", usage.total_tokens)

    # --- Raw output text ---
    output = response.choices[0].message.content
    return output

# Example:
result = absa("Staff were rude but the location was amazing. Parking was difficult but Toilets were clean.")
print("\nABSA Output:\n", result)

Input tokens : 212
Output tokens: 50
Total tokens : 262

ABSA Output:
 ```python
[
    ('staff_attitude', 'negative'),
    ('location_access', 'positive'),
    ('service_process', 'negative'),
    ('cleanliness_maintenance', 'positive')
]
```


In [ ]:
# import ast

# safe_parse(result)

### 3 Steps, 1 Supervisor
Try to do the absa using a single prompt

In [27]:
# df2 = pd.read_excel('./data/1. part_1.xlsx')
# # df2 = df2.drop(columns=(['Unnamed: 0', 'flag']))
# df2['ind'] = df2.index.astype(int)
# df2.shape

In [28]:
# import pprint

# review_id = 0

# res2 = run_absa(review=df2.loc[review_id, 'text'])

# print('place_name:', df2.loc[review_id, 'place_name_y'])
# print(20 * '*')
# print('Review: ')
# pprint.pp(df2.loc[review_id, 'text'])
# print(20 * '*')
# print('result: ')
# pprint.pp(res2)

In [ ]:
# def make_prompt(review: str) -> str:
#     return f"""Extract aspects in the review. For each aspect, assign one category: staff_attitude, service_process, food_drink, pricing_value, location_access, facilities_amenities, cleanliness_maintenance, atmosphere_vibe, natural_environment, activities_events, safety_security, policies_governance, inclusivity_community, overall_experience, other. For each category, classify the sentiment regarding that category as 'positive' | 'negative' | 'neutral'. Avoid “other” when possible. Output unique items only.
#  Review: {review}
#  Output: Pythonic list of {'category', 'sentiment'} only."""

In [ ]:
# def process_file(file: str) -> str:
#     """Read one Excel file and write its JSONL batch file."""
#     file_name = os.path.splitext(file)[0]
#     df = pd.read_excel('./data/' + file)
#     df = df.loc[:, ['place_id', 'author', 'place_name_y', 'date_parsed', 'text']]
#     df['ind'] = df.index.astype(int)
    
#     jsonl_path = './jsonl_files/' + file_name + '.jsonl'

#     with open(jsonl_path, "w", encoding="utf-8") as f:
#         for _, row in df.iterrows():
#             review = str(row["text"]).strip()

#             request = {
#                 "custom_id": f"review_{row['ind']}",
#                 "method": "POST",
#                 "url": "/v1/chat/completions",
#                 "body": {
#                     "model": "gpt-4.1-nano", 
#                     "messages": [
#                         {"role": "system", "content": "Aspect-Based Sentiment Analysis (ABSA) involves identifying specific entities (such as a person, product, service, or experience, etc.) mentioned in a text and determining the sentiment expressed toward each entity. You are an expert annotator for Aspect-Based Sentiment Analysis (ABSA) task."},
#                         {"role": "user", "content": make_prompt(review)}
#                     ],
#                     "max_tokens": 256,
#                     # "temperature": 0.0
#                 }
#             }

#             f.write(json.dumps(request, ensure_ascii=False) + "\n")
#     return file 

In [ ]:
# data_files = os.listdir('./data/')
# result_files = os.listdir('./results/')
# jsonl_files = os.listdir('./jsonl_files/')

In [32]:
# files_to_process = [f for f in data_files if f not in jsonl_files]

# max_workers = 10  

# with ThreadPoolExecutor(max_workers=max_workers) as executor:
#     futures = {executor.submit(process_file, file): file for file in files_to_process}

#     for future in tqdm(as_completed(futures), total=len(futures), desc="Processing files"):
#         file = futures[future]
#         try:
#             _ = future.result()
#         except Exception as e:
#             print(f"Error while processing {file}: {e}")

In [33]:
# batch_df.head(2)

In [ ]:
# def create_batch_for_file(file_name: str) -> str:
#     """Upload one JSONL file and create a batch for it."""
#     # Upload JSONL file
#     batch_input_file = client.files.create(
#         file=open("./jsonl_files/" + file_name, "rb"),
#         purpose="batch",
#     )
#     batch_input_file_id = batch_input_file.id

#     # Create batch
#     batch = client.batches.create(
#         input_file_id=batch_input_file_id,
#         endpoint="/v1/chat/completions",
#         completion_window="24h",
#         metadata={"description": file_name},
#     )

#     # Retrieve status once
#     # status = client.batches.retrieve(batch.id).status

#     return batch.id

In [ ]:
# def refresh_batch_status(batch_id: str) -> str:
#     b = client.batches.retrieve(batch_id)
#     return b.status

In [36]:
# def count_tokens(df: pd.DataFrame, model="gpt-4.1-nano"): # No need. Approximately 300,000
#     enc = tiktoken.encoding_for_model(model)
#     text = " ".join(df["text"].astype(str).tolist())
#     tokens = enc.encode(text)
#     return len(tokens) + 200*len(df["text"])

In [ ]:
# client = OpenAI(api_key=GPT_API_KEY)

# results = []

# APPROX_TOKENS_PER_FILE = 300_000
# QUEUE_THRESHOLD = 2_000_000 * 0.2

# queued_tokens = 0
# queued_batches = {}

# for file_name in tqdm(jsonl_files, desc="Enqueuing batches"):
#     tokens = APPROX_TOKENS_PER_FILE

#     while queued_tokens + tokens > QUEUE_THRESHOLD and queued_batches:
#         to_remove = []
#         for batch_id, info in list(queued_batches.items()):
#             status = refresh_batch_status(batch_id)
#             queued_batches[batch_id]["status"] = status

#             if status in ("completed"):
#                 queued_tokens -= info["tokens"]
#                 to_remove.append(batch_id)

#                 results.append({
#                     "file": info["file"],
#                     "batch_id": batch_id,
#                     "status": status,
#                     "description": info.get("description", info["file"]),
#                 })

#         for batch_id in to_remove:
#             queued_batches.pop(batch_id, None)

#         if queued_tokens + tokens > QUEUE_THRESHOLD:
#             time.sleep(30)


#     try:
#         batch_id = create_batch_for_file(file_name)
#         queued_batches[batch_id] = {
#             "file": file_name,
#             "tokens": tokens,
#             "status": "validating",
#             "description": file_name,
#         }
#         queued_tokens += tokens
#         print(f"\nValidating {file_name} with {tokens} tokens. Validating total: {queued_tokens}")
#     except Exception as e:
#         print(f"Error creating batch for {file_name}: {e}")

# print("\nWaiting for remaining batches to complete...")
# while queued_batches:
#     to_remove = []
#     for batch_id, info in list(queued_batches.items()):
#         status = refresh_batch_status(batch_id)
#         queued_batches[batch_id]["status"] = status

#         if status in ("completed", "failed", "cancelled", "expired"):
#             queued_tokens -= info["tokens"]
#             to_remove.append(batch_id)
#             results.append({
#                 "file": info["file"],
#                 "batch_id": batch_id,
#                 "status": status,
#                 "description": info.get("description", info["file"]),
#             })

#     for batch_id in to_remove:
#         queued_batches.pop(batch_id, None)

#     if queued_batches:
#         time.sleep(30)

# batch_df = pd.DataFrame(results, columns=["file", "batch_id", "status", "description"])
# batch_df.to_excel("batch_df.xlsx", index=False)
# print(batch_df.head(5))

Enqueuing batches:   0%|          | 0/5000 [00:00<?, ?it/s]


Validating 1. part_1.jsonl with 300000 tokens. Validating total: 300000
